# 🧬 SNP Analysis and Variant Annotation

---

## Learning Objectives

- Understand SNP types and nomenclature
- Work with VCF format
- Annotate variant consequences
- Query SNP databases

---

## SNP Types

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    VARIANT TYPES                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   SNP = Single Nucleotide Polymorphism                                  │
│   (variant present in >1% of population)                                │
│                                                                         │
│   Types by location:                                                    │
│                                                                         │
│   Gene structure:                                                       │
│   5'─────[Exon1]──intron──[Exon2]──intron──[Exon3]─────3'               │
│        ↑         ↑          ↑                      ↑                    │
│      UTR      Splice      Coding                 3'UTR                  │
│     variant    site       region                variant                 │
│                                                                         │
│   Coding region effects:                                                │
│   ┌────────────────────────────────────────────────────────────────┐    │
│   │ Type             │ Effect           │ Example                 │    │
│   ├──────────────────┼──────────────────┼─────────────────────────│    │
│   │ Synonymous       │ No AA change     │ GCA→GCG (Ala→Ala)       │    │
│   │ Missense         │ AA change        │ GCA→GTA (Ala→Val)       │    │
│   │ Nonsense         │ Stop codon       │ TGG→TGA (Trp→Stop)      │    │
│   │ Frameshift       │ Reading frame    │ Insert/delete not ×3    │    │
│   └────────────────────────────────────────────────────────────────┘    │
│                                                                         │
│   Non-coding effects:                                                   │
│   • Splice site disruption                                              │
│   • Promoter/enhancer alteration                                        │
│   • UTR regulatory changes                                              │
│   • Intronic (usually neutral)                                          │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Genetic code for variant effect prediction
CODON_TABLE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

def predict_snp_effect(ref_codon, pos_in_codon, alt_base):
    """
    Predict effect of a SNP on protein sequence.
    
    Parameters:
        ref_codon: Original codon (e.g., 'ATG')
        pos_in_codon: Position in codon (0, 1, or 2)
        alt_base: Alternative base
    
    Returns:
        Dictionary with effect information
    """
    # Create mutant codon
    mut_codon = list(ref_codon.upper())
    ref_base = mut_codon[pos_in_codon]
    mut_codon[pos_in_codon] = alt_base.upper()
    mut_codon = ''.join(mut_codon)
    
    # Translate
    ref_aa = CODON_TABLE.get(ref_codon.upper(), 'X')
    mut_aa = CODON_TABLE.get(mut_codon, 'X')
    
    # Determine effect
    if ref_aa == mut_aa:
        effect = 'synonymous'
    elif mut_aa == '*':
        effect = 'nonsense'
    elif ref_aa == '*':
        effect = 'stop_lost'
    else:
        effect = 'missense'
    
    return {
        'ref_codon': ref_codon,
        'mut_codon': mut_codon,
        'ref_aa': ref_aa,
        'mut_aa': mut_aa,
        'effect': effect,
        'notation': f"{ref_aa}→{mut_aa}" if ref_aa != mut_aa else f"{ref_aa}(syn)"
    }

# Examples
print("SNP Effect Predictions:")
print("=" * 50)

examples = [
    ('GCA', 2, 'G'),  # Synonymous
    ('GCA', 1, 'T'),  # Missense (Ala→Val)
    ('TGG', 2, 'A'),  # Nonsense (Trp→Stop)
]

for ref, pos, alt in examples:
    result = predict_snp_effect(ref, pos, alt)
    print(f"\n{ref}[{pos}]→{alt}: {result['ref_codon']}→{result['mut_codon']}")
    print(f"  Effect: {result['effect']} ({result['notation']})")

---

## VCF Format for Variants

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    VCF FORMAT                                           │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   VCF = Variant Call Format                                             │
│                                                                         │
│   ##fileformat=VCFv4.2                                                  │
│   ##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">    │
│   ##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">         │
│   #CHROM  POS     ID       REF  ALT   QUAL  FILTER  INFO  FORMAT  SAMP │
│   chr1    12345   rs123    A    G     100   PASS    AF=0.3  GT     0/1 │
│   chr1    12400   .        CT   C     80    PASS    AF=0.1  GT     0/0 │
│                                                                         │
│   Genotype encodings:                                                   │
│   ┌─────────┬──────────────────────────────────────────┐                │
│   │ 0/0     │ Homozygous reference                     │                │
│   │ 0/1     │ Heterozygous                             │                │
│   │ 1/1     │ Homozygous alternate                     │                │
│   │ 1/2     │ Heterozygous for two alts (multi-allelic)│                │
│   │ ./.     │ Missing/no call                          │                │
│   └─────────┴──────────────────────────────────────────┘                │
│                                                                         │
│   Common INFO fields:                                                   │
│   • AF: Allele frequency in population                                  │
│   • DP: Read depth                                                      │
│   • AN: Total number of alleles                                         │
│   • AC: Allele count                                                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_vcf(vcf_text):
    """
    Parse VCF text to extract variants.
    """
    variants = []
    
    for line in vcf_text.strip().split('\n'):
        if line.startswith('#'):
            continue
        
        fields = line.split('\t')
        if len(fields) >= 8:
            # Parse INFO field
            info = {}
            for item in fields[7].split(';'):
                if '=' in item:
                    key, value = item.split('=', 1)
                    info[key] = value
            
            variant = {
                'chrom': fields[0],
                'pos': int(fields[1]),
                'id': fields[2],
                'ref': fields[3],
                'alt': fields[4],
                'qual': float(fields[5]) if fields[5] != '.' else None,
                'filter': fields[6],
                'info': info
            }
            
            # Classify variant type
            if len(variant['ref']) == 1 and len(variant['alt']) == 1:
                variant['type'] = 'SNV'
            elif len(variant['ref']) < len(variant['alt']):
                variant['type'] = 'insertion'
            elif len(variant['ref']) > len(variant['alt']):
                variant['type'] = 'deletion'
            else:
                variant['type'] = 'MNV'
            
            variants.append(variant)
    
    return variants

# Example VCF
vcf_example = """##fileformat=VCFv4.2
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr1\t12345\trs123\tA\tG\t100\tPASS\tAF=0.3;DP=50
chr1\t12400\t.\tCT\tC\t80\tPASS\tAF=0.1;DP=45
chr1\t12500\trs456\tG\tGTA\t90\tPASS\tAF=0.05;DP=60"""

variants = parse_vcf(vcf_example)

print("Parsed variants:")
for v in variants:
    print(f"\n{v['chrom']}:{v['pos']} {v['id']}")
    print(f"  {v['ref']}→{v['alt']} ({v['type']})")
    print(f"  AF: {v['info'].get('AF', 'N/A')}, DP: {v['info'].get('DP', 'N/A')}")

---

## dbSNP and Variant Databases

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SNP DATABASES                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   dbSNP (NCBI):                                                         │
│   • >1 billion variants                                                 │
│   • rsID identifiers (rs12345)                                          │
│   • Population frequencies                                              │
│   • Clinical significance                                               │
│                                                                         │
│   ClinVar:                                                              │
│   • Clinical variant interpretations                                    │
│   • Pathogenic/Benign classifications                                   │
│   • Disease associations                                                │
│                                                                         │
│   gnomAD:                                                               │
│   • Population-scale variant frequencies                                │
│   • >140,000 genomes                                                    │
│   • Constraint scores                                                   │
│                                                                         │
│   rsID example: rs334 (HBB sickle cell variant)                         │
│   ┌────────────────────────────────────────────────────────────────┐    │
│   │ Position: chr11:5227002                                        │    │
│   │ Gene: HBB (Hemoglobin subunit beta)                           │    │
│   │ Change: c.20A>T (p.Glu7Val)                                    │    │
│   │ Clinical: Pathogenic (sickle cell disease)                     │    │
│   │ Global AF: 0.05 (5% in some populations)                       │    │
│   └────────────────────────────────────────────────────────────────┘    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Example variant annotation database
VARIANT_DB = {
    'rs334': {
        'chrom': 'chr11',
        'pos': 5227002,
        'gene': 'HBB',
        'ref': 'A',
        'alt': 'T',
        'hgvs_c': 'c.20A>T',
        'hgvs_p': 'p.Glu7Val',
        'consequence': 'missense_variant',
        'clinical_significance': 'Pathogenic',
        'disease': 'Sickle cell disease',
        'gnomad_af': 0.0001
    },
    'rs1800497': {
        'chrom': 'chr11',
        'pos': 113400106,
        'gene': 'ANKK1',
        'ref': 'C',
        'alt': 'T',
        'hgvs_c': 'c.2137G>A',
        'hgvs_p': 'p.Glu713Lys',
        'consequence': 'missense_variant',
        'clinical_significance': 'Benign',
        'disease': 'Dopamine receptor activity',
        'gnomad_af': 0.24
    },
    'rs7903146': {
        'chrom': 'chr10',
        'pos': 112998590,
        'gene': 'TCF7L2',
        'ref': 'C',
        'alt': 'T',
        'hgvs_c': 'c.113042C>T',
        'hgvs_p': None,  # Intronic
        'consequence': 'intron_variant',
        'clinical_significance': 'Risk factor',
        'disease': 'Type 2 diabetes',
        'gnomad_af': 0.29
    }
}

def lookup_variant(rsid):
    """Look up variant by rsID."""
    return VARIANT_DB.get(rsid)

def format_variant_report(variant_info):
    """Format variant information for display."""
    if not variant_info:
        return "Variant not found"
    
    report = f"""
Variant Report
{'=' * 50}
Location: {variant_info['chrom']}:{variant_info['pos']}
Gene: {variant_info['gene']}
Change: {variant_info['ref']}>{variant_info['alt']}
HGVS: {variant_info['hgvs_c']} / {variant_info['hgvs_p'] or 'N/A'}
Consequence: {variant_info['consequence']}
Clinical: {variant_info['clinical_significance']}
Disease: {variant_info['disease']}
gnomAD AF: {variant_info['gnomad_af']:.4f} ({variant_info['gnomad_af']*100:.2f}%)
"""
    return report

# Look up famous variants
for rsid in ['rs334', 'rs7903146']:
    info = lookup_variant(rsid)
    print(format_variant_report(info))

---

## 🏋️ Exercises

### Exercise 1: VCF Filtering
Filter VCF variants by quality and frequency.

### Exercise 2: Annotation Pipeline
Build a variant annotation workflow.

### Exercise 3: Population Analysis
Compare variant frequencies across populations.

---

## 📚 Resources

- [dbSNP](https://www.ncbi.nlm.nih.gov/snp/) - SNP database
- [ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) - Clinical variants
- [gnomAD](https://gnomad.broadinstitute.org/) - Population frequencies
- [VEP](https://www.ensembl.org/vep) - Variant Effect Predictor